### Convert Raw SBE CTD Data For DITD 2018 to Netcdf Format 

I am going through this again because the data from DITx accessed on SEANOE is interpolated in weird way which removes a lot of the short term variability. I preprocessed the CTD data incorrectly the first time because I used the wrong reference pressure ( I used zero when it should have been ~1100 ), which resulted in much higher salinities 

In [ ]:
# import packages 
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gsw

In [ ]:
raw_data = pd.read_csv('/Users/mblan/Documents/TNB_Research/data/DITD_2018_2019/SBE37SM-RS232_03715242_2019_01_03_RefPres1217.csv')

In [ ]:
raw_data

In [ ]:
# we need to add time elapsed to start time to get actual date time 
# select time elapsed columns from raw_data 
dates = raw_data.iloc[:,2]

In [ ]:
dates

In [ ]:
second_elapsed = dates.values 

In [ ]:
# time stamp from cnv data file 
starttime = np.datetime64('2018-03-09 12:00:00')

In [ ]:
seconds = np.timedelta64(second_elapsed[-1],'s')

In [ ]:
endtime = starttime + seconds

In [ ]:
dates = np.arange(starttime, endtime+np.timedelta64(180,'s'), np.timedelta64(180,'s'))

In [ ]:
dates

In [ ]:
# extract data that I want from the raw_data dataframe 
data = raw_data.iloc[:,[0,1,4,6,7,8,9,10]]

In [ ]:
data

In [ ]:
# rename variables to something easier to work with
data = data.rename(
    columns={
        "Temperature [ITS-90 deg C]": "temp",
        "Conductivity [S/m]": "con",
        "Absolute Salinity [TEOS-10 g/kg]": "abs_sal",
        "Conservative Temperature [ ITS-90 deg C]": "cons_temp",
        "Potential Density [TEOS-10 sigma-theta, kg/m^3]": "sigma0",
        "Potential Temperature [ITS-90 deg C]": "theta",
        "Practical Salinity [PSU]": "sal",
        "Pressure [Strain Gauge db]": "pres"
    }
    )

In [ ]:
data

In [ ]:
# convert data to xarray dataset 
data = data.to_xarray()

In [ ]:
# add time to dataset
data['time'] = ('index',dates)
# swap dimesnsions 
data = data.swap_dims({'index':'time'})
# remove index coordinate 
data = data.drop_vars('index')

In [ ]:
data.sal.plot(figsize=(12,4))
plt.ylim(34.78,34.88)

In [ ]:
data.temp.plot(figsize=(12,4))
plt.ylim(-1.9,-1.87)

In [ ]:
data

In [ ]:
# function to identify the line number where *END* appears in .cnv file 
# the data starts after this line so we want to identify this so we know where to start from 
def find_data_start(fname):
    with open(fname) as f:
        for i, line in enumerate(f):
            if line.strip() == "*END*":
                return i + 1
    raise ValueError("*END* not found in file")


In [ ]:
# provide file name
fname = "/Users/mblan/Documents/TNB_Research/data/DITD_2018_2019/SBE37SM-RS232_03715242_2019_01_03.cnv"
# get start line based on *END* location
start_line = find_data_start(fname)

In [ ]:
start_line

In [ ]:
# read in cnv data file, skipping appropriate number of lines, and use whitespace as delimiter
df = pd.read_csv(
    fname,
    skiprows=start_line,
    sep='\s+',
    header=None
)

In [ ]:
df

In [ ]:
ditd_bot_pres = df.iloc[:,2].values

In [ ]:
ditd_bot_pres

In [ ]:
data['pres'] = ('time',ditd_bot_pres)

In [ ]:
%matplotlib widget

In [ ]:
fig = plt.figure(figsize=(12,4))
data.sal.plot()
plt.ylim(34.8,34.9)

In [ ]:
# times to cut off at - based on zooming in above 

ditd_bot = data.sel(time=slice('2018-03-10T07:00','2019-01-02T18:00'))

In [ ]:
ditd_bot = ditd_bot.assign_coords({
    'lat': -75.28,
    'lon': 164.1,
    'z':1218
})

In [ ]:
fig = plt.figure(figsize=(12,4))
ditd_bot.temp.plot()
# plt.ylim(34.8,34.9)

### DITD Top Sensor 

In [ ]:
raw_data = pd.read_csv('/Users/mblan/Documents/TNB_Research/data/DITD_2018_2019/SBE37SM-RS232_03715273_2019_01_03_RefPre1121.csv')

In [ ]:
raw_data

In [ ]:
# we need to add time elapsed to start time to get actual date time 
# select time elapsed columns from raw_data 
dates = raw_data.iloc[:,2]

In [ ]:
dates

In [ ]:
second_elapsed = dates.values 

In [ ]:
# time stamp from cnv data file 
starttime = np.datetime64('2018-03-09 03:00:00')

In [ ]:
seconds = np.timedelta64(second_elapsed[-1],'s')

In [ ]:
endtime = starttime + seconds

In [ ]:
dates = np.arange(starttime, endtime+np.timedelta64(60,'s'), np.timedelta64(60,'s'))

In [ ]:
dates

In [ ]:
# extract data that I want from the raw_data dataframe 
data = raw_data.iloc[:,[0,1,4,6,7,8,9,10]]

In [ ]:
data

In [ ]:
# rename variables to something easier to work with
data = data.rename(
    columns={
        "Temperature [ITS-90 deg C]": "temp",
        "Conductivity [S/m]": "con",
        "Absolute Salinity [TEOS-10 g/kg]": "abs_sal",
        "Conservative Temperature [ ITS-90 deg C]": "cons_temp",
        "Potential Density [TEOS-10 sigma-theta, kg/m^3]": "sigma0",
        "Potential Temperature [ITS-90 deg C]": "theta",
        "Practical Salinity [PSU]": "sal",
        "Pressure [Strain Gauge db]": "pres"
    }
    )

In [ ]:
data

In [ ]:
# convert data to xarray dataset 
data = data.to_xarray()

In [ ]:
# add time to dataset
data['time'] = ('index',dates)
# swap dimesnsions 
data = data.swap_dims({'index':'time'})
# remove index coordinate 
data = data.drop_vars('index')

In [ ]:
plt.plot(figsize=(12,4))
data.sal.plot(figsize=(12,4))
ditd_bot.sal.plot()
plt.ylim(34.78,34.88)

In [ ]:
data.temp.plot(figsize=(12,4))
plt.ylim(-1.9,-1.87)

In [ ]:
data

In [ ]:
# function to identify the line number where *END* appears in .cnv file 
# the data starts after this line so we want to identify this so we know where to start from 
def find_data_start(fname):
    with open(fname) as f:
        for i, line in enumerate(f):
            if line.strip() == "*END*":
                return i + 1
    raise ValueError("*END* not found in file")


In [ ]:
# provide file name
fname = "/Users/mblan/Documents/TNB_Research/data/DITD_2018_2019/s1803DITD1_15273.cnv"
# get start line based on *END* location
start_line = find_data_start(fname)

In [ ]:
start_line

In [ ]:
# read in cnv data file, skipping appropriate number of lines, and use whitespace as delimiter
df = pd.read_csv(
    fname,
    skiprows=start_line,
    sep='\s+',
    header=None
)

In [ ]:
df

In [ ]:
ditd_top_pres = df.iloc[:,2].values

In [ ]:
data['pres'] = ('time',ditd_top_pres)

In [ ]:
fig = plt.figure(figsize=(12,4))
data.sal.plot()


In [ ]:
# times to cut off at - based on zooming in above 

ditd_top = data.sel(time=slice('2018-03-10T05:00','2019-01-02T19:00'))

In [ ]:
ditd_top = ditd_top.assign_coords({
    'lat': -75.28,
    'lon': 164.1,
    'z':1121
})

In [ ]:
fig = plt.figure(figsize=(12,4))
plt.plot(ditd_bot.time,ditd_bot.sal,label='1218')
plt.plot(ditd_top.time,ditd_top.sal,label='1121')
plt.legend()
plt.ylim(34.78,34.88)

### Remove Outliers 

#### DITD top CTD sensor 

In [ ]:
fig = plt.figure(figsize=(12,4))
plt.plot(ditd_top.time,ditd_top.sal)

In [ ]:
times = [
    np.datetime64('2018-04-28T03:21'),
    np.datetime64('2018-05-25T04:40'),
    np.datetime64('2018-06-08T00:16'),
    np.datetime64('2018-08-02T12:15'),
    np.datetime64('2018-08-10T09:34'),
    np.datetime64('2018-10-03T15:27'),
    np.datetime64('2018-12-18T09:19'),
]

In [ ]:
times

In [ ]:
for i in range(len(times)):
    ditd_top['sal'].loc[dict(time=times[i])] = np.nan

In [ ]:
fig = plt.figure(figsize=(12,4))
plt.plot(ditd_top.time,ditd_top.sal)

In [ ]:
# to remove
# 2018-04-28T03:21
# 2018-05-25T04:40
# 2018-06-08T00:16
# 2018-08-02T12:15
# 2018-08-10T09:34
# 2018-10-03T15:27
# 2018-12-18T09:19



#### DITD bot CTD sensor 

In [ ]:
fig = plt.figure(figsize=(12,4))
plt.plot(ditd_bot.time,ditd_bot.sal)

In [ ]:
# 2018-04-19T13:48
# 2018-07-09T21:00
# 2018-07-30T06:51
# 2018-12-05T20:18



In [ ]:
times = [
    np.datetime64('2018-04-19T13:48'),
    np.datetime64('2018-07-09T21:00'),
    np.datetime64('2018-07-30T06:51'),
    np.datetime64('2018-12-05T20:18'),
]

In [ ]:
times

In [ ]:
for i in range(len(times)):
    ditd_bot['sal'].loc[dict(time=times[i])] = np.nan

In [ ]:
fig = plt.figure(figsize=(12,4))
plt.plot(ditd_top.time,ditd_top.temp)

In [ ]:
# outliers have been removed, now we can save data 
ditd_top.to_netcdf('/Users/mblan/Documents/TNB_Research/data/DITD_2018_2019/DITD_2018_ctd_top.nc')
ditd_bot.to_netcdf('/Users/mblan/Documents/TNB_Research/data/DITD_2018_2019/DITD_2018_ctd_bot.nc')